# Gold Table Planning — Silver Schema Explorer

**Purpose:** Inspect every Silver table before designing the Gold schema.  
For each table this notebook produces:
1. A **summary block** — column names, data types, nullability, row count
2. A **pretty HTML table** rendered inline via `displayHTML()`
3. A **10-row sample** of actual data

Run this in **Microsoft Fabric** with the project Lakehouse attached.

---

| Table | Description |
|---|---|
| `silver_stg_weather` | Hourly weather per borough (Open-Meteo) |
| `silver_stg_traffic` | Hourly aggregated traffic volume per borough |
| `silver_stg_311` | Cleaned 311 complaints with SCD Type-2 change tracking |

## 0  — Setup

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ── Silver table names ────────────────────────────────────────────────────────
SILVER_TABLES = {
    "Weather":  "silver_stg_weather",
    "Traffic":  "silver_stg_traffic",
    "311":      "silver_stg_nyc_311",
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def dtype_label(spark_type):
    """Return a compact, human-readable type string."""
    name = type(spark_type).__name__
    # Strip 'Type' suffix for cleanliness
    return name.replace("Type", "") if name != "DecimalType" else str(spark_type)

def section(title):
    print("\n" + "═" * 70)
    print(f"  {title}")
    print("═" * 70)

print("✅ Setup complete — Spark session active")
print(f"   Tables to inspect: {list(SILVER_TABLES.values())}")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 3, Finished, Available, Finished, False)

✅ Setup complete — Spark session active
   Tables to inspect: ['silver_stg_weather', 'silver_stg_traffic', 'silver_stg_nyc_311']


---
## 1  — Schema Inspector Function

One function that does everything for any table:  
schema summary → HTML visualisation → row count → sample rows.

In [2]:
def inspect_table(display_name, table_name):
    """
    Full inspection of a Silver table.
    Prints a text summary, renders an HTML schema table,
    prints the row count, and shows 10 sample rows.
    """
    section(f"{display_name}  →  {table_name}")

    # ── Load ────────────────────────────────────────────────────────────────
    try:
        df = spark.table(table_name)
    except Exception as e:
        print(f"⚠️  Could not load '{table_name}': {e}")
        return

    schema = df.schema
    row_count = df.count()
    col_count = len(schema.fields)

    # ── Text summary ────────────────────────────────────────────────────────
    print(f"\n📊 Row count : {row_count:,}")
    print(f"   Columns   : {col_count}")
    print(f"\n{'─'*60}")
    print(f"  {'#':<4} {'Column Name':<35} {'Type':<18} {'Nullable'}")
    print(f"{'─'*60}")
    for i, field in enumerate(schema.fields, 1):
        nullable = "✓" if field.nullable else "✗ NOT NULL"
        print(f"  {i:<4} {field.name:<35} {dtype_label(field.dataType):<18} {nullable}")
    print(f"{'─'*60}")

    # ── Null counts per column ───────────────────────────────────────────────
    null_exprs = [F.sum(F.col(f.name).isNull().cast("int")).alias(f.name) for f in schema.fields]
    null_counts = df.select(null_exprs).collect()[0].asDict()

    has_nulls = {k: v for k, v in null_counts.items() if v and v > 0}
    if has_nulls:
        print("\n⚠️  Columns with NULLs:")
        for col_name, cnt in has_nulls.items():
            pct = 100 * cnt / row_count if row_count > 0 else 0
            print(f"     {col_name}: {cnt:,} nulls ({pct:.1f}%)")
    else:
        print("\n✅ No nulls found in any column.")

    # ── HTML schema table ────────────────────────────────────────────────────
    rows_html = ""
    for i, field in enumerate(schema.fields, 1):
        null_val = null_counts.get(field.name, 0) or 0
        null_pct = f"{100 * null_val / row_count:.1f}%" if row_count > 0 else "0%"
        null_cell = f"{null_val:,} ({null_pct})" if null_val > 0 else "<span style='color:#16a34a'>✓ None</span>"
        nullable_icon = "✓" if field.nullable else "✗"
        bg = "#f9fafb" if i % 2 == 0 else "#ffffff"
        rows_html += f"""
        <tr style='background:{bg}'>
            <td style='padding:6px 12px;color:#6b7280'>{i}</td>
            <td style='padding:6px 12px;font-weight:600;color:#111827'>{field.name}</td>
            <td style='padding:6px 12px'><code style='background:#eff6ff;color:#1d4ed8;padding:2px 6px;border-radius:4px;font-size:12px'>{dtype_label(field.dataType)}</code></td>
            <td style='padding:6px 12px;text-align:center'>{nullable_icon}</td>
            <td style='padding:6px 12px'>{null_cell}</td>
        </tr>"""

    html = f"""
    <div style='font-family:Inter,sans-serif;margin:16px 0'>
        <div style='background:#1e3a5f;color:white;padding:10px 16px;border-radius:8px 8px 0 0;display:flex;justify-content:space-between;align-items:center'>
            <span style='font-size:15px;font-weight:700'>{display_name} &nbsp;<code style='font-size:12px;opacity:0.75'>{table_name}</code></span>
            <span style='font-size:13px;opacity:0.9'>{row_count:,} rows &nbsp;|&nbsp; {col_count} columns</span>
        </div>
        <table style='width:100%;border-collapse:collapse;border:1px solid #e5e7eb;border-top:none'>
            <thead>
                <tr style='background:#f1f5f9'>
                    <th style='padding:8px 12px;text-align:left;font-size:12px;color:#6b7280;border-bottom:1px solid #e5e7eb'>#</th>
                    <th style='padding:8px 12px;text-align:left;font-size:12px;color:#6b7280;border-bottom:1px solid #e5e7eb'>Column Name</th>
                    <th style='padding:8px 12px;text-align:left;font-size:12px;color:#6b7280;border-bottom:1px solid #e5e7eb'>Data Type</th>
                    <th style='padding:8px 12px;text-align:center;font-size:12px;color:#6b7280;border-bottom:1px solid #e5e7eb'>Nullable</th>
                    <th style='padding:8px 12px;text-align:left;font-size:12px;color:#6b7280;border-bottom:1px solid #e5e7eb'>Null Count</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """
    displayHTML(html)

    # ── 10-row sample ────────────────────────────────────────────────────────
    print(f"\n📋 Sample — 10 rows from '{table_name}' (ordered by first column):")
    df.orderBy(schema.fields[0].name).limit(10).show(truncate=50, vertical=False)

print("✅ inspect_table() defined")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 4, Finished, Available, Finished, False)

✅ inspect_table() defined


---
## 2  — Weather Silver
`silver_stg_weather` — hourly weather observations per borough from the Open-Meteo API.

In [3]:
inspect_table("Weather Silver", SILVER_TABLES["Weather"])

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 5, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Weather Silver  →  silver_stg_weather
══════════════════════════════════════════════════════════════════════

📊 Row count : 13,320
   Columns   : 26

────────────────────────────────────────────────────────────
  #    Column Name                         Type               Nullable
────────────────────────────────────────────────────────────
  1    timestamp                           Timestamp          ✓
  2    temperature_2m                      Double             ✓
  3    apparent_temperature                Double             ✓
  4    relative_humidity_2m                Double             ✓
  5    dew_point_2m                        Double             ✓
  6    precipitation                       Double             ✓
  7    rain                                Double             ✓
  8    snowfall                            Double             ✓
  9    snow_depth                          Double             ✓
  10   

#,Column Name,Data Type,Nullable,Null Count
1,timestamp,Timestamp,✓,✓ None
2,temperature_2m,Double,✓,✓ None
3,apparent_temperature,Double,✓,✓ None
4,relative_humidity_2m,Double,✓,✓ None
5,dew_point_2m,Double,✓,✓ None
6,precipitation,Double,✓,✓ None
7,rain,Double,✓,✓ None
8,snowfall,Double,✓,✓ None
9,snow_depth,Double,✓,✓ None
10,wind_speed_10m,Double,✓,✓ None



📋 Sample — 10 rows from 'silver_stg_weather' (ordered by first column):
+-------------------+--------------+--------------------+--------------------+------------+-------------+----+--------+----------+--------------+------------------+--------------+-----------+------------+------------+----------------+---------------------+---------+------------------------+-------------------------+------------+-----------+----------------+---------+---------+----+
|          timestamp|temperature_2m|apparent_temperature|relative_humidity_2m|dew_point_2m|precipitation|rain|snowfall|snow_depth|wind_speed_10m|wind_direction_10m|wind_gusts_10m|cloud_cover|pressure_msl|weather_code|        timezone|timezone_abbreviation|elevation|latitude_weather_station|longitude_weather_station|     borough|snowfall_mm|weather_category| latitude|longitude|year|
+-------------------+--------------+--------------------+--------------------+------------+-------------+----+--------+----------+--------------+------------

---
## 3  — Traffic Silver
`silver_stg_traffic` — hourly aggregated traffic volume per borough, derived from 15-minute sensor readings.

In [14]:
inspect_table("Traffic Silver", SILVER_TABLES["Traffic"])

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 16, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Traffic Silver  →  silver_stg_traffic
══════════════════════════════════════════════════════════════════════

📊 Row count : 696
   Columns   : 5

────────────────────────────────────────────────────────────
  #    Column Name                         Type               Nullable
────────────────────────────────────────────────────────────
  1    borough                             String             ✓
  2    hour                                Timestamp          ✓
  3    traffic_volume                      Long               ✓
  4    sensor_readings                     Long               ✓
  5    sensor_count                        Long               ✓
────────────────────────────────────────────────────────────

✅ No nulls found in any column.


#,Column Name,Data Type,Nullable,Null Count
1,borough,String,✓,✓ None
2,hour,Timestamp,✓,✓ None
3,traffic_volume,Long,✓,✓ None
4,sensor_readings,Long,✓,✓ None
5,sensor_count,Long,✓,✓ None



📋 Sample — 10 rows from 'silver_stg_traffic' (ordered by first column):
+-------+-------------------+--------------+---------------+------------+
|borough|               hour|traffic_volume|sensor_readings|sensor_count|
+-------+-------------------+--------------+---------------+------------+
|  Bronx|2026-02-01 22:00:00|           297|             16|           4|
|  Bronx|2026-01-20 00:00:00|           190|             16|           4|
|  Bronx|2026-02-01 23:00:00|           297|             16|           4|
|  Bronx|2026-01-20 01:00:00|           114|             16|           4|
|  Bronx|2026-02-02 00:00:00|           219|             16|           4|
|  Bronx|2026-01-20 02:00:00|            88|             16|           4|
|  Bronx|2026-02-02 01:00:00|           165|             16|           4|
|  Bronx|2026-01-20 03:00:00|            91|             16|           4|
|  Bronx|2026-02-02 02:00:00|           140|             16|           4|
|  Bronx|2026-01-20 04:00:00|          

---
## 4  — 311 Complaints Silver
`silver_stg_311` — cleaned 311 service requests with SCD Type-2 change tracking (`change_date`).

In [5]:
inspect_table("311 Silver", SILVER_TABLES["311"])

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 7, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  311 Silver  →  silver_stg_nyc_311
══════════════════════════════════════════════════════════════════════

📊 Row count : 1,165,545
   Columns   : 43

────────────────────────────────────────────────────────────
  #    Column Name                         Type               Nullable
────────────────────────────────────────────────────────────
  1    address_type                        String             ✓
  2    agency                              String             ✓
  3    agency_name                         String             ✓
  4    bbl                                 String             ✓
  5    borough                             String             ✓
  6    bridge_highway_direction            String             ✓
  7    bridge_highway_name                 String             ✓
  8    bridge_highway_segment              String             ✓
  9    city                                String             ✓
  10   c

#,Column Name,Data Type,Nullable,Null Count
1,address_type,String,✓,"6,713 (0.6%)"
2,agency,String,✓,✓ None
3,agency_name,String,✓,✓ None
4,bbl,String,✓,"137,962 (11.8%)"
5,borough,String,✓,✓ None
6,bridge_highway_direction,String,✓,"1,160,356 (99.6%)"
7,bridge_highway_name,String,✓,"1,158,232 (99.4%)"
8,bridge_highway_segment,String,✓,"1,158,228 (99.4%)"
9,city,String,✓,"55,772 (4.8%)"
10,closed_date,Timestamp,✓,"98,933 (8.5%)"



📋 Sample — 10 rows from 'silver_stg_nyc_311' (ordered by first column):
+------------+------+-------------------------------+----+-------------+---------------------------+-------------------+--------------------------------------------------+----+-------------------+---------------+-----------------+----------------+-------------------+--------------+--------------+-------------------+------------+----------------+------------+---------------------+---------------------+--------+-----------------+-------------+------------------+----------------------+-------------+------------------+---------------+------------------------------+--------------------------------------------------+---------+------+-----------+----------+-----------------------------------+-----------------+----------------------------------------+--------------------------+-----------+-------------------+-----------+
|address_type|agency|                    agency_name| bbl|      borough|   bridge_highway_direction|br

---
## 5  — Cross-Table Summary

A side-by-side view of row counts, date ranges, and the join keys to verify alignment before building Gold.

In [6]:
section("Cross-Table Summary — Join Key Alignment Check")

summaries = []

# ── Weather ──────────────────────────────────────────────────────────────────
try:
    df_w = spark.table(SILVER_TABLES["Weather"])
    # Weather join keys: borough + timestamp (truncated to hour)
    # Identify the timestamp column — usually called 'timestamp'
    ts_col = "timestamp"  # adjust if your column has a different name
    w_stats = df_w.agg(
        F.count("*").alias("rows"),
        F.countDistinct("borough").alias("boroughs"),
        F.min(ts_col).alias("earliest"),
        F.max(ts_col).alias("latest"),
    ).collect()[0]
    w_boroughs = [r[0] for r in df_w.select("borough").distinct().orderBy("borough").collect()]
    summaries.append(("Weather", w_stats["rows"], w_stats["boroughs"], w_stats["earliest"], w_stats["latest"], w_boroughs))
except Exception as e:
    summaries.append(("Weather", None, None, None, None, [str(e)]))

# ── Traffic ──────────────────────────────────────────────────────────────────
try:
    df_t = spark.table(SILVER_TABLES["Traffic"])
    # Traffic join key: borough + hour
    t_ts_col = "hour"  # aggregated hourly column — adjust if named differently
    t_stats = df_t.agg(
        F.count("*").alias("rows"),
        F.countDistinct("borough").alias("boroughs"),
        F.min(t_ts_col).alias("earliest"),
        F.max(t_ts_col).alias("latest"),
    ).collect()[0]
    t_boroughs = [r[0] for r in df_t.select("borough").distinct().orderBy("borough").collect()]
    summaries.append(("Traffic", t_stats["rows"], t_stats["boroughs"], t_stats["earliest"], t_stats["latest"], t_boroughs))
except Exception as e:
    summaries.append(("Traffic", None, None, None, None, [str(e)]))

# ── 311 ──────────────────────────────────────────────────────────────────────
try:
    df_3 = spark.table(SILVER_TABLES["311"])
    # 311 join key: borough + date_trunc('hour', created_date)
    c_ts_col = "created_date"  # adjust if named differently
    c_stats = df_3.agg(
        F.count("*").alias("rows"),
        F.countDistinct("borough").alias("boroughs"),
        F.min(c_ts_col).alias("earliest"),
        F.max(c_ts_col).alias("latest"),
    ).collect()[0]
    c_boroughs = [r[0] for r in df_3.select("borough").distinct().orderBy("borough").collect()]
    summaries.append(("311", c_stats["rows"], c_stats["boroughs"], c_stats["earliest"], c_stats["latest"], c_boroughs))
except Exception as e:
    summaries.append(("311", None, None, None, None, [str(e)]))

# ── Print alignment table ────────────────────────────────────────────────────
print(f"\n{'─'*80}")
print(f"  {'Source':<12} {'Rows':>10}  {'Boroughs':>9}  {'Earliest':<22} {'Latest'}")
print(f"{'─'*80}")
for name, rows, boros, earliest, latest, blist in summaries:
    row_str = f"{rows:,}" if rows is not None else "ERROR"
    boros_str = str(boros) if boros is not None else "?"
    earliest_str = str(earliest) if earliest else "?"
    latest_str = str(latest) if latest else "?"
    print(f"  {name:<12} {row_str:>10}  {boros_str:>9}  {earliest_str:<22} {latest_str}")
print(f"{'─'*80}")

print("\n🗺️  Borough values found in each table:")
for name, *_, blist in summaries:
    print(f"   {name}: {blist}")

print("""
─────────────────────────────────────────────────────────────────────────────
⚠️  JOIN KEY CHECKLIST — confirm before building Gold:

  [ ] All 3 tables have 5 boroughs
  [ ] Borough strings use the SAME format (title case: Manhattan, Brooklyn…)
  [ ] Date ranges overlap — ideally all cover 2026-01-01 to ~today
  [ ] Weather timestamp column can be truncated to hour → matches Traffic.hour
  [ ] 311 created_date can be truncated to hour → matches Weather/Traffic
─────────────────────────────────────────────────────────────────────────────
""")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 8, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Cross-Table Summary — Join Key Alignment Check
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────────────────────
  Source             Rows   Boroughs  Earliest               Latest
────────────────────────────────────────────────────────────────────────────────
  Weather          13,320          5  2026-01-01 00:00:00    2026-04-21 23:00:00
  Traffic           2,088          4  2026-01-03 00:00:00    2026-02-03 23:00:00
  311           1,165,545          5  2026-01-01 00:00:00    2026-04-19 01:50:47
────────────────────────────────────────────────────────────────────────────────

🗺️  Borough values found in each table:
   Weather: ['bronx', 'brooklyn', 'manhattan', 'queens', 'statenisland']
   Traffic: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens']
   311: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']

───────────

---
## 6  — Borough Name Format Check

The Gold join key is `borough + hour`. This cell shows **exactly** what string values each table contains for `borough`, so you can spot format mismatches before the join fails silently.

In [7]:
section("Borough Name Format Check")

tables_and_cols = [
    ("Weather",  SILVER_TABLES["Weather"],  "borough"),
    ("Traffic",  SILVER_TABLES["Traffic"],  "borough"),
    ("311",      SILVER_TABLES["311"],      "borough"),
]

# Collect all borough values and their occurrence counts
all_borough_vals = {}
for name, tbl, col in tables_and_cols:
    try:
        vals = (
            spark.table(tbl)
            .groupBy(col)
            .count()
            .orderBy(col)
            .collect()
        )
        all_borough_vals[name] = {r[col]: r["count"] for r in vals}
        print(f"\n  {name} → '{col}' values:")
        for bname, cnt in all_borough_vals[name].items():
            print(f"       {repr(bname):30s}  {cnt:>10,} rows")
    except Exception as e:
        print(f"  {name} — ERROR: {e}")
        all_borough_vals[name] = {}

# Detect mismatches
print("\n")
all_sets = [set(v.keys()) for v in all_borough_vals.values()]
if len(all_sets) == 3:
    common = all_sets[0] & all_sets[1] & all_sets[2]
    union  = all_sets[0] | all_sets[1] | all_sets[2]
    if common == union:
        print("✅ Borough values are IDENTICAL across all 3 tables — Gold join will work.")
    else:
        print("❌ Borough value MISMATCH detected!")
        print(f"   Common across all 3 : {sorted(common)}")
        for name, vals in all_borough_vals.items():
            only_in = set(vals.keys()) - common
            if only_in:
                print(f"   Only in {name}        : {sorted(only_in)}")
        print("\n   ➤  Fix: apply F.initcap(F.regexp_replace('borough', ' ', '')) in Gold join cell.")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 9, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Borough Name Format Check
══════════════════════════════════════════════════════════════════════

  Weather → 'borough' values:
       'bronx'                              2,664 rows
       'brooklyn'                           2,664 rows
       'manhattan'                          2,664 rows
       'queens'                             2,664 rows
       'statenisland'                       2,664 rows

  Traffic → 'borough' values:
       'Bronx'                              1,008 rows
       'Brooklyn'                             504 rows
       'Manhattan'                            504 rows
       'Queens'                                72 rows

  311 → 'borough' values:
       'Bronx'                            248,043 rows
       'Brooklyn'                         356,228 rows
       'Manhattan'                        221,777 rows
       'Queens'                           284,089 rows
       'Staten Island'   

---
## 7  — Cardinality & Value Distribution Snapshots

Quick look at the distribution of key categorical columns — useful for planning Gold aggregations.

In [8]:
section("311 — Complaint Type Distribution (Top 20)")
try:
    # 311 complaint type column — adjust name if different
    complaint_col = "complaint_type"  # snake_case after Silver cleaning
    (
        spark.table(SILVER_TABLES["311"])
        .groupBy(complaint_col)
        .count()
        .orderBy(F.desc("count"))
        .limit(20)
        .show(truncate=60)
    )
except Exception as e:
    print(f"Skipped: {e}")

section("311 — Status Distribution")
try:
    (
        spark.table(SILVER_TABLES["311"])
        .groupBy("status")
        .count()
        .orderBy(F.desc("count"))
        .show()
    )
except Exception as e:
    print(f"Skipped: {e}")

section("Weather — weather_category Distribution")
try:
    (
        spark.table(SILVER_TABLES["Weather"])
        .groupBy("weather_category")
        .count()
        .orderBy(F.desc("count"))
        .show()
    )
except Exception as e:
    print(f"Skipped: {e}")

section("Traffic — sensor_count Distribution")
try:
    (
        spark.table(SILVER_TABLES["Traffic"])
        .groupBy("sensor_count")
        .count()
        .orderBy("sensor_count")
        .show()
    )
except Exception as e:
    print(f"Skipped: {e}")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 10, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  311 — Complaint Type Distribution (Top 20)
══════════════════════════════════════════════════════════════════════
+------------------------+------+
|          complaint_type| count|
+------------------------+------+
|         Illegal Parking|179947|
|          HEAT/HOT WATER|179356|
|     Noise - Residential|107015|
|        Blocked Driveway| 62072|
|             Snow or Ice| 55216|
|        Street Condition| 39232|
|    UNSANITARY CONDITION| 36912|
|                PLUMBING| 28990|
| Noise - Street/Sidewalk| 27999|
|           PAINT/PLASTER| 21304|
|            Water System| 21270|
|         Dirty Condition| 20482|
|       Abandoned Vehicle| 19683|
|Traffic Signal Condition| 19528|
|             DOOR/WINDOW| 17367|
|                   Noise| 17364|
|      Noise - Commercial| 15539|
|              WATER LEAK| 15309|
|                 GENERAL| 12252|
|         Noise - Vehicle| 12053|
+------------------------+----

---
## 8  — Numeric Range Summary

Min / Max / Mean for the numeric columns that will be carried into Gold — confirms no leftover outliers after Silver cleaning.

In [9]:
section("Weather — Numeric Column Ranges")
try:
    WEATHER_NUMERIC = [
        "temperature_2m", "apparent_temperature", "dew_point_2m",
        "relative_humidity_2m", "rain", "snowfall", "snow_depth",
        "wind_speed_10m", "wind_gusts_10m", "cloud_cover",
        "precipitation", "pressure_msl"
    ]
    df_w = spark.table(SILVER_TABLES["Weather"])
    # Only describe columns that actually exist in this table
    existing = [c for c in WEATHER_NUMERIC if c in df_w.columns]
    df_w.select(existing).describe().show(truncate=False)
except Exception as e:
    print(f"Skipped: {e}")

section("Traffic — Numeric Column Ranges")
try:
    TRAFFIC_NUMERIC = ["traffic_volume", "sensor_readings", "sensor_count"]
    df_t = spark.table(SILVER_TABLES["Traffic"])
    existing_t = [c for c in TRAFFIC_NUMERIC if c in df_t.columns]
    df_t.select(existing_t).describe().show(truncate=False)
except Exception as e:
    print(f"Skipped: {e}")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 11, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Weather — Numeric Column Ranges
══════════════════════════════════════════════════════════════════════
+-------+-----------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+-----------------+-------------------+------------------+
|summary|temperature_2m   |apparent_temperature|dew_point_2m       |relative_humidity_2m|rain                |snowfall            |snow_depth          |wind_speed_10m    |wind_gusts_10m    |cloud_cover      |precipitation      |pressure_msl      |
+-------+-----------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+------------------+-----------------+-------------------+------------------+
|count  |13320            |13320               |13320              |13320      

---
## 9  — Overlap Row Count (Preview of Gold Join Size)

Runs the exact join that Gold will use and reports how many rows will be produced **before** any 311 aggregation. This is the key sanity check: zero rows = borough mismatch or date range gap.

In [10]:
section("Gold Join Preview — Row Count Estimate")

try:
    df_w = spark.table(SILVER_TABLES["Weather"])
    df_t = spark.table(SILVER_TABLES["Traffic"])

    # ── Normalise borough to lowercase for join (adjust if already standardised) ──
    df_w2 = df_w.withColumn("borough_key", F.lower(F.trim(F.col("borough"))))
    df_t2 = df_t.withColumn("borough_key", F.lower(F.trim(F.col("borough"))))

    # ── Truncate both timestamps to hour ─────────────────────────────────────
    # Weather: assumes column named 'timestamp' (TimestampType)
    # Traffic: assumes column named 'hour'  (TimestampType)
    df_w2 = df_w2.withColumn("hour_key", F.date_trunc("hour", F.col("timestamp")))
    df_t2 = df_t2.withColumn("hour_key", F.date_trunc("hour", F.col("hour")))

    joined = df_w2.join(df_t2, on=["borough_key", "hour_key"], how="inner")
    join_count = joined.count()

    print(f"\n  Weather rows     : {df_w.count():>10,}")
    print(f"  Traffic rows     : {df_t.count():>10,}")
    print(f"  ── Inner join ──")
    print(f"  Joined rows      : {join_count:>10,}")

    if join_count == 0:
        print("\n  ❌ Zero rows — check borough format + timestamp alignment!")
    elif join_count == min(df_w.count(), df_t.count()):
        print("\n  ✅ Full join — all rows matched.")
    else:
        pct = 100 * join_count / df_w.count()
        print(f"\n  ⚠️  Partial join ({pct:.1f}% of weather rows matched) — some hours have no traffic data.")
        print("     This is normal if traffic sensor coverage doesn't match weather hours.")

    # ── Sample of what Gold rows will look like ───────────────────────────────
    print("\n  Sample joined rows (first 5):")
    joined.select(
        "borough_key", "hour_key",
        "temperature_2m", "weather_category",
        "traffic_volume", "sensor_count"
    ).limit(5).show(truncate=False)

except Exception as e:
    print(f"\n  Could not run join preview: {e}")
    print("  Run sections 2–4 first to confirm column names, then adjust ts_col / hour_col above.")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 12, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Gold Join Preview — Row Count Estimate
══════════════════════════════════════════════════════════════════════

  Weather rows     :     13,320
  Traffic rows     :      2,088
  ── Inner join ──
  Joined rows      :      2,088

  ✅ Full join — all rows matched.

  Sample joined rows (first 5):
+-----------+-------------------+--------------+----------------+--------------+------------+
|borough_key|hour_key           |temperature_2m|weather_category|traffic_volume|sensor_count|
+-----------+-------------------+--------------+----------------+--------------+------------+
|bronx      |2026-01-20 00:00:00|-4.2          |Clear           |190           |4           |
|bronx      |2026-01-20 00:00:00|-4.2          |Clear           |190           |4           |
|bronx      |2026-01-20 00:00:00|-4.2          |Clear           |190           |4           |
|bronx      |2026-01-20 01:00:00|-5.3          |Clear           |114

---
## 10  — Output: Column Name Reference Card

Prints a compact column reference for all three tables — copy this into the Gold planning session.

In [11]:
section("Column Reference Card — All Silver Tables")

for display_name, table_name in SILVER_TABLES.items():
    try:
        fields = spark.table(table_name).schema.fields
        print(f"\n  ┌─ {display_name} ({table_name}) ──────────────────────────────────")
        for f in fields:
            print(f"  │  {f.name:<40} {dtype_label(f.dataType)}")
        print(f"  └{'─'*60}")
    except Exception as e:
        print(f"  {display_name}: ERROR — {e}")

print("""
═══════════════════════════════════════════════════════════════════════════════
  ✅ Schema exploration complete.
  Next step → Gold table schema planning:
    - Confirm join keys: borough + hour
    - Decide which columns carry forward vs get aggregated
    - Define 311 complaint count aggregation grain (per borough+hour+type)
    - Decide Gold grain: one row per borough+hour, or borough+hour+complaint_type
═══════════════════════════════════════════════════════════════════════════════
""")

StatementMeta(, f1cdc4b1-b8a4-492d-be67-abfa2dc676eb, 13, Finished, Available, Finished, False)


══════════════════════════════════════════════════════════════════════
  Column Reference Card — All Silver Tables
══════════════════════════════════════════════════════════════════════

  ┌─ Weather (silver_stg_weather) ──────────────────────────────────
  │  timestamp                                Timestamp
  │  temperature_2m                           Double
  │  apparent_temperature                     Double
  │  relative_humidity_2m                     Double
  │  dew_point_2m                             Double
  │  precipitation                            Double
  │  rain                                     Double
  │  snowfall                                 Double
  │  snow_depth                               Double
  │  wind_speed_10m                           Double
  │  wind_direction_10m                       Double
  │  wind_gusts_10m                           Double
  │  cloud_cover                              Double
  │  pressure_msl                             Doubl